# Step 08: Merge All Info

Input:

    - Publication, Journal, Gender, Affiliation
        - data/interim/publication/6_first_author_institution_info.csv
        - data/interim/publication/6_last_author_institution_info.csv

        - data/interim/publication/7_first_author_country_info.csv
        - data/interim/publication/7_last_author_country_info.csv

        - data/interim/publication/5_first_author_publication_info.csv
        - data/interim/publication/5_last_author_publication_info.csv

    - Journal List
        - data/raw/journal/3_JournalStatus.xlsx   



Output:
    - data/processed/1_first_{CI}alldata.csv
    - data/processed/1_last_{CI}alldata.csv

    - data/processed/2_first_{CI}journal.csv
    - data/processed/2_last_{CI}journal.csv

    - data/processed/2_first_{CI}institution.csv
    - data/processed/2_last_{CI}institution.csv
       


Purpose:
    - Merge all information
    - Aggregate and collapse to journal and institution level data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [ ]:
# -----------------------------
# Paths
# -----------------------------


PROJECT_ROOT = Path(__file__).resolve().parents[2]

ALL_PUB_FILE = PROJECT_ROOT / "data" / "interim" / "publication"/ "1_AllPublications.csv"

FIRST_JRN_IN = PROJECT_ROOT / "data" / "interim" / "publication"/ "5_first_author_publication_info.csv"
LAST_JRN_IN = PROJECT_ROOT / "data" / "interim" / "publication"/ "5_last_author_publication_info.csv"

FIRST_INS_IN = PROJECT_ROOT / "data" / "interim" / "publication"/ "6_first_author_institution_info.csv"
LAST_INS_IN = PROJECT_ROOT / "data" / "interim" / "publication"/ "6_last_author_institution_info.csv"

FIRST_CNY_IN = PROJECT_ROOT / "data" / "interim" / "publication"/ "7_first_author_country_info.csv"
LAST_CNY_IN = PROJECT_ROOT / "data" / "interim" / "publication"/ "7_last_author_country_info.csv"


JRN_LIST = PROJECT_ROOT / "data" / "raw" / "journal" / "3_JournalStatus.xlsx"


FILE_OUT = PROJECT_ROOT / "data" / "processed" 
FILE_OUT.mkdir(parents=True, exist_ok=True)



# -----------------------------
# Gender Confidence Threshold
# -----------------------------

CI = 0.9  # replace with 0.7 for sensitivity analysis

# 1 First authorship

### 1.1 Filter journals and gender classification confidence

In [ ]:
first = pd.read_csv(FIRST_JRN_IN)
print(first.columns)
first

unique_journals = first['Source Title'].nunique()
print("Number of unique journal:", unique_journals)

unique_country = first['country'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = first['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

In [ ]:
# Filter journal ranking list
standard = pd.read_excel(JRN_LIST)
print(standard.columns)
standard


# Merge on journal name and publication year
merged = first.merge(
    standard,
    how="left",
    left_on=["Source Title"],
    right_on=["Journal name"]
)

merged = merged.drop(columns=["Journal name"])

# total observations
total_obs = merged.shape[0]

# number of observations with either_10% = 1
count_either = merged["either_10%"].sum(skipna=True)

# percentage over total
percent_either = count_either / total_obs * 100

print(f"Total observations: {total_obs}")
print(f"Observations with either_10% = 1: {count_either}")
print(f"Percentage: {percent_either:.2f}%")

# unique journals with either_10% = 1
unique_journals = merged.loc[merged["either_10%"] == 1, "Source Title"].nunique()

print(f"Unique 'Source Title' with either_10% = 1: {unique_journals}")

# Step 1: Keep only rows where either_10% = 1
first = merged[merged["either_10%"] == 1].copy()
cols_to_drop = ["all_10%", "either_10%", "merge", "editor_name_either", "editor_gender_either","editor_ratio_either","editor_ratio_all","annual"]
first = first.drop(columns=cols_to_drop, errors="ignore")

del merged

In [ ]:
unique_journals = first['Source Title'].nunique()
print("Number of unique journal:", unique_journals)

unique_country = first['country'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = first['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

In [ ]:
first = first.drop(['pre gender'], axis=1)

def assign_gender(proba):
    if proba >= CI:
        return 1  # 女性
    elif (1 - proba) >= CI:
        return 0  # 男性
    else:
        return np.nan  # 不确定

first['female_author'] = first['proba'].apply(assign_gender)

In [ ]:
# 统计数量
gender_counts = first['female_author'].value_counts(dropna=False)

# 统计占比（百分比）
gender_percentages = first['female_author'].value_counts(dropna=False, normalize=True) * 100

# 创建汇总 DataFrame
gender_summary = pd.DataFrame({
    '数量': gender_counts,
    '占比(%)': gender_percentages.round(2)
})

# 重命名索引以便更清晰
gender_summary.index = gender_summary.index.map({
    1.0: '女性',
    0.0: '男性',
    np.nan: '不确定'
})

print(gender_summary)

# 去掉不确定性别样本
    # 去掉 gender 列为 NaN 的行
first_clean = first.dropna(subset=['female_author'])

# 显示每列的非空值数量和占比
total_rows = len(first_clean)
non_na_stats = pd.DataFrame({
    '非空值数量': first_clean.notna().sum(),
    '非空值占比(%)': (first_clean.notna().sum() / total_rows * 100).round(2)
})
print(non_na_stats)

In [ ]:
unique_journals = first_clean['Source Title'].nunique()
print("Number of unique journal:", unique_journals)

unique_country = first_clean['country'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = first_clean['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

### 1.2 Merge institution and country information

In [ ]:
# Path to your CSV file
first_ins = pd.read_csv(FIRST_INS_IN)
print(first_ins.columns)
print(first_ins.head())

# Path to your CSV file
first_cny = pd.read_csv(FIRST_CNY_IN)
print(first_cny.columns)
print(first_cny.head())

In [ ]:
first_ins = first_ins.drop(['affiliation_qs','affiliation_the','affiliation_oa','Publication Year'], axis=1)
first_cny = first_cny.drop(['Publication Year','affiliation'], axis=1)

In [ ]:
first_clean = pd.merge(first_clean,first_ins,on='index_num',how='left')
first_clean = first_clean.drop(['affiliation_y'], axis=1)
first_clean = first_clean.rename(columns={'affiliation_x':'affiliation'})

first_clean = pd.merge(first_clean,first_cny,on='index_num',how='left')
first_clean = first_clean.drop(['country_y'], axis=1)
first_clean = first_clean.rename(columns={'country_x':'country'})
first_clean.columns

# 显示每列的非空值数量和占比
total_rows = len(first_clean)
non_na_stats = pd.DataFrame({
    '非空值数量': first_clean.notna().sum(),
    '非空值占比(%)': (first_clean.notna().sum() / total_rows * 100).round(2)
})
# 使用 to_string() 确保完整显示
print(non_na_stats.to_string())

### 1.3 Generate other indicators

In [ ]:
first_clean = first_clean.rename(
    columns={'female_editor_ratio':'jrn_female_edirto',
             'has_female_editor':'jrn_female_edihas',
             'total_author_num':'pap_author_num',
             'Source Title':'journal'})
first_clean.columns

In [ ]:
# Journal level

# 1) 每年每本期刊的文章数量
first_clean['jrn_sumwork'] = first_clean.groupby(['Publication Year', 'journal'])['journal'].transform('size')

# 2) 每年每本期刊的平均被引数量
first_clean['jrn_avgcited'] = first_clean.groupby(['Publication Year', 'journal'])['pap_cited'].transform('mean')

# 3) 发达国家作者发表文章占比
developed_sum = first_clean.groupby(['Publication Year', 'journal'])['cny_developed'].transform(lambda x: (x == 1).sum())
first_clean['jrn_developed'] = developed_sum / first_clean['jrn_sumwork']

# 4) 生成 ins_top 变量
first_clean['ins_top'] = np.where(
    (first_clean['ins_qs'] <= 200) | 
    (first_clean['ins_arwu'] <= 200) | 
    (first_clean['ins_therank'] <= 200), 1, 0
)

# 5) 排名前200学校作者发表文章占比
top_sum = first_clean.groupby(['Publication Year', 'journal'])['ins_top'].transform('sum')
first_clean['jrn_top'] = top_sum / first_clean['jrn_sumwork']

# 检查是否有缺失值
print("缺失值检查:")
print(first_clean[['jrn_sumwork', 'jrn_avgcited', 'jrn_developed', 'ins_top', 'jrn_top']].isnull().sum())

# 设置pandas显示选项，禁用科学计数法
pd.set_option('display.float_format', '{:.2f}'.format)

# 查看统计摘要
print("\n统计摘要:")
print(first_clean[['jrn_sumwork', 'jrn_avgcited', 'jrn_developed', 'ins_top', 'jrn_top']].describe())

In [ ]:
# field level
# 1) 每年每个领域的平均团队作者数量
first_clean['fld_author_num'] = first_clean.groupby(['Publication Year', 'field'])['pap_author_num'].transform('mean')

#2) 每年每个领域里有女性编辑的期刊占比
first_clean['fld_total_journal'] = first_clean.groupby(['Publication Year', 'field'])['journal'].transform('nunique')

# 计算每年每个领域中有女性编辑的期刊数量
# 先创建一个临时列，标记每本期刊是否有女性编辑
first_clean['journal_has_female_editor'] = first_clean.groupby(['Publication Year', 'field', 'journal'])['jrn_female_edihas'].transform('max')

# 计算每年每个领域中有女性编辑的期刊数量
female_editor_count = first_clean.groupby(['Publication Year', 'field'])['journal_has_female_editor'].transform(lambda x: (x == 1).sum())

# 计算占比
first_clean['fld_female_editor'] = female_editor_count / first_clean['fld_total_journal']

first_clean = first_clean.drop(['journal_has_female_editor'], axis=1)


# 设置pandas显示选项，禁用科学计数法
pd.set_option('display.float_format', '{:.2f}'.format)

# 查看统计摘要
print("\n统计摘要:")
print(first_clean[['fld_sumwork', 'fld_avgcited', 'fld_author_num', 'fld_female_editor']].describe())

In [ ]:
# institution level


# 1) 每年每机构的平均被引数量
first_clean['ins_avgcited'] = first_clean.groupby(['Publication Year', 'affiliation'])['pap_cited'].transform('mean')



# 生成学科相关指标

# 1) 计算每年每机构的总论文数
first_clean['ins_sumwork'] = first_clean.groupby(['Publication Year', 'affiliation'])['affiliation'].transform('count')

# 2) 生成 ins_stem_work - STEM领域论文占比
first_clean['stem_flag'] = (first_clean['stem'] == 1).astype(int)
stem_count = first_clean.groupby(['Publication Year', 'affiliation'])['stem_flag'].transform('sum')
first_clean['ins_stem_work'] = stem_count / first_clean['ins_sumwork']

# 3) 生成 ins_tech_work - 科技领域（category=5）论文占比
first_clean['tech_flag'] = (first_clean['category'] == 5).astype(int)
tech_count = first_clean.groupby(['Publication Year', 'affiliation'])['tech_flag'].transform('sum')
first_clean['ins_tech_work'] = tech_count / first_clean['ins_sumwork']

# 4) 生成 ins_ssah_work - 人文社科领域（category=1 or 4）论文占比
first_clean['ssah_flag'] = first_clean['category'].isin([1, 4]).astype(int)
ssah_count = first_clean.groupby(['Publication Year', 'affiliation'])['ssah_flag'].transform('sum')
first_clean['ins_ssah_work'] = ssah_count / first_clean['ins_sumwork']

# 5) 生成 ins_bio_work - 生物领域（category=2）论文占比
first_clean['bio_flag'] = (first_clean['category'] == 2).astype(int)
bio_count = first_clean.groupby(['Publication Year', 'affiliation'])['bio_flag'].transform('sum')
first_clean['ins_bio_work'] = bio_count / first_clean['ins_sumwork']

# 6) 生成 ins_phy_work - 物理领域（category=3）论文占比
first_clean['phy_flag'] = (first_clean['category'] == 3).astype(int)
phy_count = first_clean.groupby(['Publication Year', 'affiliation'])['phy_flag'].transform('sum')
first_clean['ins_phy_work'] = phy_count / first_clean['ins_sumwork']


# 处理NaN值
first_clean['ins_stem_work'] = first_clean['ins_stem_work'].fillna(0)
first_clean['ins_tech_work'] = first_clean['ins_tech_work'].fillna(0)
first_clean['ins_ssah_work'] = first_clean['ins_ssah_work'].fillna(0)
first_clean['ins_bio_work'] = first_clean['ins_bio_work'].fillna(0)
first_clean['ins_phy_work'] = first_clean['ins_phy_work'].fillna(0)

# 删除临时标志列
first_clean = first_clean.drop(['stem_flag', 'tech_flag', 'ssah_flag','bio_flag','phy_flag'], axis=1)

# 设置pandas显示选项，禁用科学计数法
pd.set_option('display.float_format', '{:.2f}'.format)

# 查看统计摘要
print("\n统计摘要:")
print(first_clean[['ins_sumwork', 'ins_avgcited', 'ins_stem_work', 'ins_tech_work','ins_ssah_work','ins_bio_work','ins_phy_work']].describe())

In [ ]:
first_clean.to_csv(FILE_OUT / "1_first_{CI}alldata.csv", index=False)

### 1.4 Aggregate to journal level

In [ ]:
columns = ['index_num', 'journal', 'Publication Year', 'cny_developed', 'ins_top',
           'field', 'category', 'wos_id', 'stem', 
           'fld_sumwork', 'fld_avgcited', 'fld_author_num', 'fld_total_journal', 'fld_female_editor',
           'jrn_female_edirto', 'jrn_female_edihas', 'jrn_sjr', 'jrn_hindex', 'jrn_totref', 'jrn_refperdoc', 'jrn_female_ratio',
           'jrn_top','jrn_sumwork', 'jrn_avgcited', 'jrn_developed',            
           'female_author']

journal = first_clean[columns]

unique_journals = journal['journal'].nunique()
print("Number of unique journal:", unique_journals)

unique_journals = first_clean['journal'].nunique()
print("Number of unique journal:", unique_journals)

In [ ]:
# collapse and generate indicators

# --- 1) Field–year proportions: fld_top, fld_developed ---

journal['fld_top'] = (
    journal.groupby(['field', 'Publication Year'])['ins_top']
        .transform('mean')
)
journal['fld_developed'] = (
    journal.groupby(['field', 'Publication Year'])['cny_developed']
        .transform('mean')
)

journal = journal.drop(columns=['cny_developed','ins_top'])


# --- 2) Select/ensure columns for downstream steps ---
# (Keep your specified columns + the new fld_top, fld_developed, and ins_top if needed)
keep_cols = [
     'journal', 'Publication Year',
    'field', 'category', 'wos_id', 'stem',
    'fld_sumwork', 'fld_avgcited', 'fld_author_num', 'fld_total_journal', 'fld_female_editor','fld_top', 'fld_developed',
    'jrn_female_edirto', 'jrn_female_edihas', 'jrn_sjr', 'jrn_hindex', 'jrn_totref',
    'jrn_refperdoc', 'jrn_female_ratio', 'jrn_top', 'jrn_sumwork', 'jrn_avgcited', 'jrn_developed',
    'female_author'
]
# Keep only those that actually exist to avoid KeyErrors if any are missing
keep_cols = [c for c in keep_cols if c in journal.columns]
journal = journal[keep_cols].copy()



# --- 3) Collapse to journal–year level ---
group_keys = ['journal', 'Publication Year']

categorical_cols = [c for c in ['field', 'category', 'wos_id', 'stem'] if c in journal.columns]

# Numeric features to average (exclude group keys, index_num, and categoricals)
exclude = set(group_keys + categorical_cols)
numeric_cols = [c for c in journal.columns if c not in exclude]


agg_dict = {col: 'mean' for col in numeric_cols}
# For categoricals, keep the first (should be constant within each journal)
for c in categorical_cols:
    agg_dict[c] = 'first'

collapsed = (
    journal.groupby(group_keys, as_index=False)
        .agg(agg_dict)
        .rename(columns={'female_author': 'female_ratio'})
)



unique_journals = collapsed['journal'].nunique()
print("Number of unique journal:", unique_journals)
print(collapsed.head())

In [ ]:
collapsed.to_csv(FILE_OUT / "2_first_{CI}journal.csv", index=False)

### 1.5 Aggregate to institution level

In [ ]:
columns = ['Publication Year',  'country_ins', 'country_code','affiliation', 'cny_developed', 'ins_top',
           'ins_avgcited', 'ins_sumwork', 'ins_stem_work', 'ins_tech_work', 'ins_ssah_work', 'ins_bio_work', 'ins_phy_work',
           'ins_arwu','ins_qs', 'ins_AcademicRep', 'ins_CiteFaculty', 'ins_EmployerRep', 'ins_therank', 'ins_the_citations', 'ins_the_indinc',
           'ins_the_intlotl', 'ins_the_research', 'ins_the_teaching', 'ins_oa_wrkcnt', 'ins_oa_ctdbycnt',
           'cny_gni_per_capita', 'cny_life_expectancy', 'cny_birth_rate', 'cny_female_employment', 'cny_female_edu_expectancy','cny_fertility',
           'cny_gini', 'cny_gov_edu_expenditure', 'cny_female_tertiary_grad', 'cny_female_tertiary_enrol', 'cny_school_year',
           'cny_female_school_year', 'cny_male_school_year', 'cny_female_labor_participation', 'cny_female_parliament', 'cny_female_secondary_edu', 
           'cny_gii', 'cny_gov_acc', 'cny_gov_corr', 'cny_gov_eff', 'cny_gov_law', 'cny_gov_reg', 'cny_gov_sta', 'cny_fragile_state', 
           'cny_gender_development', 'cny_female_hdi', 'cny_human_inequality', 'cny_hdi', 'cny_democracy', 'cny_RD_expenditure',
           'cny_female_researcher_gov', 'cny_female_researcher_edu', 'cny_female_researcher', 'cny_researcher_percentage', 'cny_female_employ_agr',
           'cny_female_employ_ind', 'cny_female_employ_ser', 'cny_female_lfp_adv', 'cny_female_lfp_bas', 'cny_female_parental_leave',
           'cny_female_infant_death', 'cny_gender_lfp_ratio', 'cny_female_primary_enrol', 'cny_female_secondary_enrol', 'cny_gender_bir_ratio',
           'cny_female_unemploy_adv', 'cny_female_unemploy_bas', 'cny_female_unemploy', 'cny_wbl', 'cny_diff_edu_attain', 'cny_diff_employ_agr', 
           'cny_diff_employ_ind', 'cny_diff_employ_ser', 'cny_diff_employ', 'cny_diff_edu_expectancy', 'cny_diff_teriary_grad', 'cny_diff_lfp', 
           'cny_diff_lfp_adv', 'cny_diff_lfp_bas', 'cny_diff_parental_leave', 'cny_diff_infant_death', 'cny_diff_primary_enrol', 'cny_diff_secondary_enrol',
           'cny_diff_tertiary_enrol', 'cny_diff_une_adv', 'cny_diff_une_bas', 'cny_diff_une', 'cny_gdp_capita',    
           'female_author']

institution = first_clean[columns]

unique_country = institution['country_ins'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = institution['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

In [ ]:
# collapse 


# --- Keys and column sets ---
group_keys = ["affiliation", "Publication Year"]
cat_cols   = [c for c in ["country_ins", "country_code"] if c in institution.columns]

for c in institution.columns:
    if c not in group_keys + cat_cols:
        institution[c] = pd.to_numeric(institution[c], errors="coerce")

# Build aggregation: mean for numerics (incl. female_author), first for categoricals
numeric_cols = [c for c in institution.columns if c not in group_keys + cat_cols]
agg_dict = {c: "mean" for c in numeric_cols}
for c in cat_cols:
    agg_dict[c] = "first"

# Collapse to affiliation–year level
collapsed = (
    institution
    .groupby(group_keys, as_index=False)
    .agg(agg_dict)
    .rename(columns={"female_author": "female_ratio"})
)


unique_country = collapsed['country_ins'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = collapsed['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

In [ ]:
# --- Define ordering pieces ---
group_keys = ["affiliation", "Publication Year"]
cat_cols   = ["country_ins", "country_code"]  # keep only those that exist
cat_cols   = [c for c in cat_cols if c in collapsed.columns]

preferred_numeric_order = [
    "ins_arwu","ins_qs","ins_AcademicRep","ins_CiteFaculty","ins_EmployerRep",
    "ins_therank","ins_the_citations","ins_the_indinc","ins_the_intlotl",
    "ins_the_research","ins_the_teaching","ins_oa_wrkcnt","ins_oa_ctdbycnt",
    "ins_top","ins_avgcited","ins_sumwork","ins_stem_work","ins_tech_work",
    "ins_ssah_work","ins_bio_work","ins_phy_work",
    "cny_life_expectancy","cny_birth_rate","cny_fertility","cny_female_infant_death",
    "cny_gender_bir_ratio","cny_developed","cny_gni_per_capita","cny_gdp_capita",
    "cny_female_employment","cny_female_labor_participation","cny_female_hdi","cny_hdi",
    "cny_female_employ_agr","cny_female_employ_ind","cny_female_employ_ser",
    "cny_female_lfp_adv","cny_female_lfp_bas","cny_gender_lfp_ratio",
    "cny_female_unemploy_adv","cny_female_unemploy_bas","cny_female_unemploy",
    "cny_gov_edu_expenditure","cny_female_edu_expectancy","cny_female_tertiary_grad",
    "cny_female_tertiary_enrol","cny_school_year","cny_female_school_year",
    "cny_male_school_year","cny_female_secondary_edu","cny_female_primary_enrol",
    "cny_female_secondary_enrol","cny_RD_expenditure","cny_female_researcher_gov",
    "cny_female_researcher_edu","cny_female_researcher","cny_researcher_percentage",
    "cny_gini","cny_gii","cny_gender_development","cny_human_inequality",
    "cny_diff_edu_attain","cny_diff_employ_agr","cny_diff_employ_ind",
    "cny_diff_employ_ser","cny_diff_employ","cny_diff_edu_expectancy",
    "cny_diff_teriary_grad","cny_diff_lfp","cny_diff_lfp_adv","cny_diff_lfp_bas",
    "cny_diff_parental_leave","cny_diff_infant_death","cny_diff_primary_enrol",
    "cny_diff_secondary_enrol","cny_diff_tertiary_enrol","cny_diff_une_adv",
    "cny_diff_une_bas","cny_diff_une","cny_female_parliament","cny_gov_acc",
    "cny_gov_corr","cny_gov_eff","cny_gov_law","cny_gov_reg","cny_gov_sta",
    "cny_fragile_state","cny_democracy","cny_female_parental_leave","cny_wbl"
]

# Keep only columns that actually exist (no KeyErrors), and avoid duplicates
existing = set(collapsed.columns)
ordered_numeric = [c for c in preferred_numeric_order if c in existing]

# Build the final order
final_order = (
    [c for c in group_keys if c in existing]
    + cat_cols
    + (["female_ratio"] if "female_ratio" in existing else [])
    + [c for c in ordered_numeric
       if c not in group_keys + cat_cols + ["female_ratio"]]
)

# Append any remaining columns (if any) to not lose data
remaining = [c for c in collapsed.columns if c not in final_order]
final_order += remaining

# Reorder
collapsed = collapsed[final_order]

# (Optional) quick report
missing_from_pref = [c for c in preferred_numeric_order if c not in existing]
if missing_from_pref:
    print("These preferred columns were not found and were skipped:", missing_from_pref)


In [ ]:
collapsed.to_csv(FILE_OUT / "2_first_{CI}institution.csv", index=False)

# 2 Last Corresponding authorship

### 2.1 Filter journal and classified gender

In [ ]:
last = pd.read_csv(LAST_JRN_IN)
print(last.columns)


unique_journals = last['Source Title'].nunique()
print("Number of unique journal:", unique_journals)

unique_country = last['country'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = last['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

In [ ]:
merged = last.merge(
    standard,
    how="left",
    left_on=["Source Title"],
    right_on=["Journal name"]
)

# If you don't need the duplicate Journal_name column anymore:
merged = merged.drop(columns=["Journal name"])

# total observations
total_obs = merged.shape[0]

# number of observations with either_10% = 1
count_either = merged["either_10%"].sum(skipna=True)

# percentage over total
percent_either = count_either / total_obs * 100

print(f"Total observations: {total_obs}")
print(f"Observations with either_10% = 1: {count_either}")
print(f"Percentage: {percent_either:.2f}%")

# unique journals with either_10% = 1
unique_journals = merged.loc[merged["either_10%"] == 1, "Source Title"].nunique()

print(f"Unique 'Source Title' with either_10% = 1: {unique_journals}")

# Step 1: Keep only rows where either_10% = 1
last = merged[merged["either_10%"] == 1].copy()

cols_to_drop = ["all_10%", "either_10%", "merge", "editor_name_either", "editor_gender_either","editor_ratio_either","editor_ratio_all","annual"]
last = last.drop(columns=cols_to_drop, errors="ignore")


print("Shape after filtering:", last.shape)
print(last.columns)

del merged

In [ ]:
unique_journals = last['Source Title'].nunique()
print("Number of unique journal:", unique_journals)

unique_country = last['country'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = last['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

In [ ]:
### Gender classification
last = last.drop(['pre gender'], axis=1)

def assign_gender(proba):
    if proba >= CI:
        return 1  # 女性
    elif (1 - proba) >= CI:
        return 0  # 男性
    else:
        return np.nan  # 不确定

last['female_author'] = last['proba'].apply(assign_gender)

In [ ]:
# 统计数量
gender_counts = last['female_author'].value_counts(dropna=False)

# 统计占比（百分比）
gender_percentages = last['female_author'].value_counts(dropna=False, normalize=True) * 100

# 创建汇总 DataFrame
gender_summary = pd.DataFrame({
    '数量': gender_counts,
    '占比(%)': gender_percentages.round(2)
})

# 重命名索引以便更清晰
gender_summary.index = gender_summary.index.map({
    1.0: '女性',
    0.0: '男性',
    np.nan: '不确定'
})

print(gender_summary)

# 去掉不确定性别样本
    # 去掉 gender 列为 NaN 的行
last_clean = last.dropna(subset=['female_author'])

# 显示每列的非空值数量和占比
total_rows = len(last_clean)
non_na_stats = pd.DataFrame({
    '非空值数量': last_clean.notna().sum(),
    '非空值占比(%)': (last_clean.notna().sum() / total_rows * 100).round(2)
})
print(non_na_stats)

In [ ]:
unique_journals = last_clean['Source Title'].nunique()
print("Number of unique journal:", unique_journals)

unique_country = last_clean['country'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = last_clean['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

### 2.2 Merge Institution and Country Infomration

In [ ]:
# Path to your CSV file
last_ins = pd.read_csv(LAST_INS_IN)
print(last_ins.columns)
print(last_ins.head())

# Path to your CSV file
last_cny = pd.read_csv(LAST_CNY_IN)
print(last_cny.columns)
print(last_cny.head())

In [ ]:
last_ins = last_ins.drop(['affiliation_qs','affiliation_the','affiliation_oa','Publication Year'], axis=1)
last_cny = last_cny.drop(['Publication Year','affiliation'], axis=1)

last_clean = pd.merge(last_clean,last_ins,on='index_num',how='left')
last_clean = last_clean.drop(['affiliation_y'], axis=1)
last_clean = last_clean.rename(columns={'affiliation_x':'affiliation'})

last_clean = pd.merge(last_clean,last_cny,on='index_num',how='left')
last_clean = last_clean.drop(['country_y'], axis=1)
last_clean = last_clean.rename(columns={'country_x':'country'})
last_clean.columns

In [ ]:
 # 显示每列的非空值数量和占比
total_rows = len(last_clean)
non_na_stats = pd.DataFrame({
    '非空值数量': last_clean.notna().sum(),
    '非空值占比(%)': (last_clean.notna().sum() / total_rows * 100).round(2)
})
# 使用 to_string() 确保完整显示
print(non_na_stats.to_string())

### 2.3 Generate other indicators

In [ ]:
last_clean = last_clean.rename(
    columns={'female_editor_ratio':'jrn_female_edirto',
             'has_female_editor':'jrn_female_edihas',
             'total_author_num':'pap_author_num',
             'Source Title':'journal'})
last_clean.columns

In [ ]:
# Journal level

# 1) 每年每本期刊的文章数量
last_clean['jrn_sumwork'] = last_clean.groupby(['Publication Year', 'journal'])['journal'].transform('size')

# 2) 每年每本期刊的平均被引数量
last_clean['jrn_avgcited'] = last_clean.groupby(['Publication Year', 'journal'])['pap_cited'].transform('mean')

# 3) 发达国家作者发表文章占比
developed_sum = last_clean.groupby(['Publication Year', 'journal'])['cny_developed'].transform(lambda x: (x == 1).sum())
last_clean['jrn_developed'] = developed_sum / last_clean['jrn_sumwork']

# 4) 生成 ins_top 变量
last_clean['ins_top'] = np.where(
    (last_clean['ins_qs'] <= 200) | 
    (last_clean['ins_arwu'] <= 200) | 
    (last_clean['ins_therank'] <= 200), 1, 0
)

# 5) 排名前200学校作者发表文章占比
top_sum = last_clean.groupby(['Publication Year', 'journal'])['ins_top'].transform('sum')
last_clean['jrn_top'] = top_sum / last_clean['jrn_sumwork']

# 检查是否有缺失值
print("缺失值检查:")
print(last_clean[['jrn_sumwork', 'jrn_avgcited', 'jrn_developed', 'ins_top', 'jrn_top']].isnull().sum())

# 设置pandas显示选项，禁用科学计数法
pd.set_option('display.float_format', '{:.2f}'.format)

# 查看统计摘要
print("\n统计摘要:")
print(last_clean[['jrn_sumwork', 'jrn_avgcited', 'jrn_developed', 'ins_top', 'jrn_top']].describe())

In [ ]:
# field level
# 1) 每年每个领域的平均团队作者数量
last_clean['fld_author_num'] = last_clean.groupby(['Publication Year', 'field'])['pap_author_num'].transform('mean')

#2) 每年每个领域里有女性编辑的期刊占比
last_clean['fld_total_journal'] = last_clean.groupby(['Publication Year', 'field'])['journal'].transform('nunique')

# 计算每年每个领域中有女性编辑的期刊数量
# 先创建一个临时列，标记每本期刊是否有女性编辑
last_clean['journal_has_female_editor'] = last_clean.groupby(['Publication Year', 'field', 'journal'])['jrn_female_edihas'].transform('max')

# 计算每年每个领域中有女性编辑的期刊数量
female_editor_count = last_clean.groupby(['Publication Year', 'field'])['journal_has_female_editor'].transform(lambda x: (x == 1).sum())

# 计算占比
last_clean['fld_female_editor'] = female_editor_count / last_clean['fld_total_journal']

last_clean = last_clean.drop(['journal_has_female_editor'], axis=1)


# 查看统计摘要
print("\n统计摘要:")
print(last_clean[['fld_sumwork', 'fld_avgcited', 'fld_author_num', 'fld_female_editor']].describe())

In [ ]:
# institution level


# 1) 每年每机构的平均被引数量
last_clean['ins_avgcited'] = last_clean.groupby(['Publication Year', 'affiliation'])['pap_cited'].transform('mean')



# 生成学科相关指标

# 1) 计算每年每机构的总论文数
last_clean['ins_sumwork'] = last_clean.groupby(['Publication Year', 'affiliation'])['affiliation'].transform('count')

# 2) 生成 ins_stem_work - STEM领域论文占比
last_clean['stem_flag'] = (last_clean['stem'] == 1).astype(int)
stem_count = last_clean.groupby(['Publication Year', 'affiliation'])['stem_flag'].transform('sum')
last_clean['ins_stem_work'] = stem_count / last_clean['ins_sumwork']

# 3) 生成 ins_tech_work - 科技领域（category=5）论文占比
last_clean['tech_flag'] = (last_clean['category'] == 5).astype(int)
tech_count = last_clean.groupby(['Publication Year', 'affiliation'])['tech_flag'].transform('sum')
last_clean['ins_tech_work'] = tech_count / last_clean['ins_sumwork']

# 4) 生成 ins_ssah_work - 人文社科领域（category=1 or 4）论文占比
last_clean['ssah_flag'] = last_clean['category'].isin([1, 4]).astype(int)
ssah_count = last_clean.groupby(['Publication Year', 'affiliation'])['ssah_flag'].transform('sum')
last_clean['ins_ssah_work'] = ssah_count / last_clean['ins_sumwork']

# 5) 生成 ins_bio_work - 生物领域（category=2）论文占比
last_clean['bio_flag'] = (last_clean['category'] == 2).astype(int)
bio_count = last_clean.groupby(['Publication Year', 'affiliation'])['bio_flag'].transform('sum')
last_clean['ins_bio_work'] = bio_count / last_clean['ins_sumwork']

# 6) 生成 ins_phy_work - 物理领域（category=3）论文占比
last_clean['phy_flag'] = (last_clean['category'] == 3).astype(int)
phy_count = last_clean.groupby(['Publication Year', 'affiliation'])['phy_flag'].transform('sum')
last_clean['ins_phy_work'] = phy_count / last_clean['ins_sumwork']


# 处理NaN值
last_clean['ins_stem_work'] = last_clean['ins_stem_work'].fillna(0)
last_clean['ins_tech_work'] = last_clean['ins_tech_work'].fillna(0)
last_clean['ins_ssah_work'] = last_clean['ins_ssah_work'].fillna(0)
last_clean['ins_bio_work'] = last_clean['ins_bio_work'].fillna(0)
last_clean['ins_phy_work'] = last_clean['ins_phy_work'].fillna(0)

# 删除临时标志列
last_clean = last_clean.drop(['stem_flag', 'tech_flag', 'ssah_flag','bio_flag','phy_flag'], axis=1)

# 设置pandas显示选项，禁用科学计数法
pd.set_option('display.float_format', '{:.2f}'.format)

# 查看统计摘要
print("\n统计摘要:")
print(last_clean[['ins_sumwork', 'ins_avgcited', 'ins_stem_work', 'ins_tech_work','ins_ssah_work','ins_bio_work','ins_phy_work']].describe())

In [ ]:
last_clean.to_csv(FILE_OUT / "1_last_{CI}alldata.csv", index=False)

### 2.4 aggregate to journal level

In [ ]:
columns = ['index_num', 'journal', 'Publication Year', 'cny_developed', 'ins_top',
           'field', 'category', 'wos_id', 'stem', 
           'fld_sumwork', 'fld_avgcited', 'fld_author_num', 'fld_total_journal', 'fld_female_editor',
           'jrn_female_edirto', 'jrn_female_edihas', 'jrn_sjr', 'jrn_hindex', 'jrn_totref', 'jrn_refperdoc', 'jrn_female_ratio',
           'jrn_top','jrn_sumwork', 'jrn_avgcited', 'jrn_developed',            
           'female_author']

journal = last_clean[columns]

unique_journals = journal['journal'].nunique()
print("Number of unique journal:", unique_journals)

unique_journals = last_clean['journal'].nunique()
print("Number of unique journal:", unique_journals)

In [ ]:
# collapse and generate indicators

# --- 1) Field–year proportions: fld_top, fld_developed ---

journal['fld_top'] = (
    journal.groupby(['field', 'Publication Year'])['ins_top']
        .transform('mean')
)
journal['fld_developed'] = (
    journal.groupby(['field', 'Publication Year'])['cny_developed']
        .transform('mean')
)

journal = journal.drop(columns=['cny_developed','ins_top'])


# --- 2) Select/ensure columns for downstream steps ---
# (Keep your specified columns + the new fld_top, fld_developed, and ins_top if needed)
keep_cols = [
     'journal', 'Publication Year',
    'field', 'category', 'wos_id', 'stem',
    'fld_sumwork', 'fld_avgcited', 'fld_author_num', 'fld_total_journal', 'fld_female_editor','fld_top', 'fld_developed',
    'jrn_female_edirto', 'jrn_female_edihas', 'jrn_sjr', 'jrn_hindex', 'jrn_totref',
    'jrn_refperdoc', 'jrn_female_ratio', 'jrn_top', 'jrn_sumwork', 'jrn_avgcited', 'jrn_developed',
    'female_author'
]
# Keep only those that actually exist to avoid KeyErrors if any are missing
keep_cols = [c for c in keep_cols if c in journal.columns]
journal = journal[keep_cols].copy()



# --- 3) Collapse to journal–year level ---
group_keys = ['journal', 'Publication Year']

categorical_cols = [c for c in ['field', 'category', 'wos_id', 'stem'] if c in journal.columns]

# Numeric features to average (exclude group keys, index_num, and categoricals)
exclude = set(group_keys + categorical_cols)
numeric_cols = [c for c in journal.columns if c not in exclude]


agg_dict = {col: 'mean' for col in numeric_cols}
# For categoricals, keep the first (should be constant within each journal)
for c in categorical_cols:
    agg_dict[c] = 'first'

collapsed = (
    journal.groupby(group_keys, as_index=False)
        .agg(agg_dict)
        .rename(columns={'female_author': 'female_ratio'})
)



unique_journals = collapsed['journal'].nunique()
print("Number of unique journal:", unique_journals)
print(collapsed.head())

In [ ]:
unique_journals = collapsed['journal'].nunique()
print("Number of unique journal:", unique_journals)

collapsed.to_csv(FILE_OUT / "2_last_{CI}journal.csv", index=False)

### 2.5 Aggregate to Institution Level

In [ ]:
columns = ['Publication Year',  'country_ins', 'country_code','affiliation', 'cny_developed', 'ins_top',
           'ins_avgcited', 'ins_sumwork', 'ins_stem_work', 'ins_tech_work', 'ins_ssah_work', 'ins_bio_work', 'ins_phy_work',
           'ins_arwu','ins_qs', 'ins_AcademicRep', 'ins_CiteFaculty', 'ins_EmployerRep', 'ins_therank', 'ins_the_citations', 'ins_the_indinc',
           'ins_the_intlotl', 'ins_the_research', 'ins_the_teaching', 'ins_oa_wrkcnt', 'ins_oa_ctdbycnt',
           'cny_gni_per_capita', 'cny_life_expectancy', 'cny_birth_rate', 'cny_female_employment', 'cny_female_edu_expectancy','cny_fertility',
           'cny_gini', 'cny_gov_edu_expenditure', 'cny_female_tertiary_grad', 'cny_female_tertiary_enrol', 'cny_school_year',
           'cny_female_school_year', 'cny_male_school_year', 'cny_female_labor_participation', 'cny_female_parliament', 'cny_female_secondary_edu', 
           'cny_gii', 'cny_gov_acc', 'cny_gov_corr', 'cny_gov_eff', 'cny_gov_law', 'cny_gov_reg', 'cny_gov_sta', 'cny_fragile_state', 
           'cny_gender_development', 'cny_female_hdi', 'cny_human_inequality', 'cny_hdi', 'cny_democracy', 'cny_RD_expenditure',
           'cny_female_researcher_gov', 'cny_female_researcher_edu', 'cny_female_researcher', 'cny_researcher_percentage', 'cny_female_employ_agr',
           'cny_female_employ_ind', 'cny_female_employ_ser', 'cny_female_lfp_adv', 'cny_female_lfp_bas', 'cny_female_parental_leave',
           'cny_female_infant_death', 'cny_gender_lfp_ratio', 'cny_female_primary_enrol', 'cny_female_secondary_enrol', 'cny_gender_bir_ratio',
           'cny_female_unemploy_adv', 'cny_female_unemploy_bas', 'cny_female_unemploy', 'cny_wbl', 'cny_diff_edu_attain', 'cny_diff_employ_agr', 
           'cny_diff_employ_ind', 'cny_diff_employ_ser', 'cny_diff_employ', 'cny_diff_edu_expectancy', 'cny_diff_teriary_grad', 'cny_diff_lfp', 
           'cny_diff_lfp_adv', 'cny_diff_lfp_bas', 'cny_diff_parental_leave', 'cny_diff_infant_death', 'cny_diff_primary_enrol', 'cny_diff_secondary_enrol',
           'cny_diff_tertiary_enrol', 'cny_diff_une_adv', 'cny_diff_une_bas', 'cny_diff_une', 'cny_gdp_capita',    
           'female_author']

institution = last_clean[columns]

unique_country = institution['country_ins'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = institution['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

In [ ]:
# collapse 


# --- Keys and column sets ---
group_keys = ["affiliation", "Publication Year"]
cat_cols   = [c for c in ["country_ins", "country_code"] if c in institution.columns]

for c in institution.columns:
    if c not in group_keys + cat_cols:
        institution[c] = pd.to_numeric(institution[c], errors="coerce")

# Build aggregation: mean for numerics (incl. female_author), first for categoricals
numeric_cols = [c for c in institution.columns if c not in group_keys + cat_cols]
agg_dict = {c: "mean" for c in numeric_cols}
for c in cat_cols:
    agg_dict[c] = "first"

# Collapse to affiliation–year level
collapsed = (
    institution
    .groupby(group_keys, as_index=False)
    .agg(agg_dict)
    .rename(columns={"female_author": "female_ratio"})
)


unique_country = collapsed['country_ins'].nunique()
print("Number of unique country:", unique_country)

unique_affiliation = collapsed['affiliation'].nunique()
print("Number of unique affiliation:", unique_affiliation)

In [ ]:
# --- Define ordering pieces ---
group_keys = ["affiliation", "Publication Year"]
cat_cols   = ["country_ins", "country_code"]  # keep only those that exist
cat_cols   = [c for c in cat_cols if c in collapsed.columns]

preferred_numeric_order = [
    "ins_arwu","ins_qs","ins_AcademicRep","ins_CiteFaculty","ins_EmployerRep",
    "ins_therank","ins_the_citations","ins_the_indinc","ins_the_intlotl",
    "ins_the_research","ins_the_teaching","ins_oa_wrkcnt","ins_oa_ctdbycnt",
    "ins_top","ins_avgcited","ins_sumwork","ins_stem_work","ins_tech_work",
    "ins_ssah_work","ins_bio_work","ins_phy_work",
    "cny_life_expectancy","cny_birth_rate","cny_fertility","cny_female_infant_death",
    "cny_gender_bir_ratio","cny_developed","cny_gni_per_capita","cny_gdp_capita",
    "cny_female_employment","cny_female_labor_participation","cny_female_hdi","cny_hdi",
    "cny_female_employ_agr","cny_female_employ_ind","cny_female_employ_ser",
    "cny_female_lfp_adv","cny_female_lfp_bas","cny_gender_lfp_ratio",
    "cny_female_unemploy_adv","cny_female_unemploy_bas","cny_female_unemploy",
    "cny_gov_edu_expenditure","cny_female_edu_expectancy","cny_female_tertiary_grad",
    "cny_female_tertiary_enrol","cny_school_year","cny_female_school_year",
    "cny_male_school_year","cny_female_secondary_edu","cny_female_primary_enrol",
    "cny_female_secondary_enrol","cny_RD_expenditure","cny_female_researcher_gov",
    "cny_female_researcher_edu","cny_female_researcher","cny_researcher_percentage",
    "cny_gini","cny_gii","cny_gender_development","cny_human_inequality",
    "cny_diff_edu_attain","cny_diff_employ_agr","cny_diff_employ_ind",
    "cny_diff_employ_ser","cny_diff_employ","cny_diff_edu_expectancy",
    "cny_diff_teriary_grad","cny_diff_lfp","cny_diff_lfp_adv","cny_diff_lfp_bas",
    "cny_diff_parental_leave","cny_diff_infant_death","cny_diff_primary_enrol",
    "cny_diff_secondary_enrol","cny_diff_tertiary_enrol","cny_diff_une_adv",
    "cny_diff_une_bas","cny_diff_une","cny_female_parliament","cny_gov_acc",
    "cny_gov_corr","cny_gov_eff","cny_gov_law","cny_gov_reg","cny_gov_sta",
    "cny_fragile_state","cny_democracy","cny_female_parental_leave","cny_wbl"
]

# Keep only columns that actually exist (no KeyErrors), and avoid duplicates
existing = set(collapsed.columns)
ordered_numeric = [c for c in preferred_numeric_order if c in existing]

# Build the final order
final_order = (
    [c for c in group_keys if c in existing]
    + cat_cols
    + (["female_ratio"] if "female_ratio" in existing else [])
    + [c for c in ordered_numeric
       if c not in group_keys + cat_cols + ["female_ratio"]]
)

# Append any remaining columns (if any) to not lose data
remaining = [c for c in collapsed.columns if c not in final_order]
final_order += remaining

# Reorder
collapsed = collapsed[final_order]

# (Optional) quick report
missing_from_pref = [c for c in preferred_numeric_order if c not in existing]
if missing_from_pref:
    print("These preferred columns were not found and were skipped:", missing_from_pref)


In [ ]:
collapsed.to_csv(FILE_OUT / "2_last_{CI}institution.csv", index=False)